In [1]:
import os
import sys

sys.path.append("..")

import torch

from src.ood.registry import get_ood_detectors

In [2]:
# ============================================================
# Cell 2 - Load Model & Dataset
# ============================================================

import torch
import torchvision.transforms as transforms
import torchvision.datasets as datasets

from torchvision.models import resnet18
from torch.utils.data import DataLoader

# ------------------------------------------------------------
# Device
# ------------------------------------------------------------

device = torch.device(
    "mps" if torch.backends.mps.is_available() else "cpu"
)

print("Device:", device)

# ------------------------------------------------------------
# Model
# ------------------------------------------------------------

model = resnet18(weights=None)

model.fc = torch.nn.Linear(
    model.fc.in_features,
    10
)

checkpoint = torch.load(
    "../models/resnet18_cifar10.pth",
    map_location=device
)

model.load_state_dict(checkpoint)

model.to(device)

model.eval()

print("Model Loaded Successfully")

# ------------------------------------------------------------
# CIFAR10 Transform
# ------------------------------------------------------------

transform = transforms.Compose([

    transforms.ToTensor(),

    transforms.Normalize(
        (0.4914, 0.4822, 0.4465),
        (0.2023, 0.1994, 0.2010)
    )

])

# ------------------------------------------------------------
# Dataset
# ------------------------------------------------------------

test_dataset = datasets.CIFAR10(

    root="../data",

    train=False,

    download=True,

    transform=transform

)

test_loader = DataLoader(

    test_dataset,

    batch_size=1,

    shuffle=False

)

print("Dataset Loaded Successfully")

Device: mps
Model Loaded Successfully
Dataset Loaded Successfully


In [4]:
# ============================================================
# Cell 3 - Neural State Collector
# ============================================================

from src.utils.neural_state_collector import NeuralStateCollector

# Last convolution block of ResNet18
target_layer = model.layer4[-1]

collector = NeuralStateCollector(target_layer)

collector.register_hooks()

print("Neural State Collector Ready")

Neural State Collector Ready


In [5]:
# ============================================================
# Cell 4 - Test MSP Detector
# ============================================================

images, labels = next(iter(test_loader))

images = images.to(device)

states = collector.collect(model, images)

output = states["logits"]

sample = {

    "model": model,

    "input_tensor": images,

    "output": output

}

for detector in get_ood_detectors():

    print("=" * 60)

    print(detector.name)

    print("=" * 60)

    result = detector.detect(**sample)

    for key, value in result.items():

        print(f"{key:<30}: {value}")

collector.remove_hooks()

MSP Detector
msp_probability               : 0.6937355399131775
msp_ood_score                 : 0.3062644600868225
